<a href="https://colab.research.google.com/github/xyingg1/ist1314-chunkz/blob/main/big_data_pandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Setup
!pip install -q pandas scikit-learn matplotlib
import pandas as pd, numpy as np, time, tracemalloc, json
print("Ready.")


Ready.


In [ ]:
from google.colab import files
uploaded = files.upload()


Saving archive (7).zip to archive (7).zip


In [ ]:
#load data into tables
orders = pd.read_csv("olist_orders_dataset.csv",
                      parse_dates=["order_purchase_timestamp", "order_delivered_customer_date",
                                   "order_estimated_delivery_date"]) #tell pandas these column are dates not plain text
items = pd.read_csv("olist_order_items_dataset.csv")
products = pd.read_csv("olist_products_dataset.csv")
reviews = pd.read_csv("olist_order_reviews_dataset.csv")
print(orders.shape, items.shape, products.shape, reviews.shape)

(99441, 8) (112650, 7) (32951, 9) (99224, 7)


In [ ]:
#clean and join
# 4. Clean & join
def clean_and_join(orders, items, products, reviews):
    delivered = orders[orders["order_status"] == "delivered"].copy()
    delivered["delivery_delay_days"] = (
        delivered["order_delivered_customer_date"] - delivered["order_estimated_delivery_date"]
    ).dt.total_seconds() / 86400.0
#on="order_id" means match rows where the order_id is the same in both tables.
#how="inner" means only keep rows that have a match in both tables
    df = delivered.merge(items, on="order_id", how="inner")
    df = df.merge(products, on="product_id", how="inner")
    df = df.merge(reviews, on="order_id", how="inner")
    #.astype(int) converts True→1 and False→0
    df["is_bad_review"] = (df["review_score"] <= 3).astype(int)

    return df[["order_id", "product_category_name", "delivery_delay_days",
               "is_bad_review", "review_score", "price", "freight_value"]]

t0 = time.perf_counter()
base_df = clean_and_join(orders, items, products, reviews)
join_time = time.perf_counter() - t0
print(f"join+clean time: {join_time:.3f}s, rows: {len(base_df):,}")


join+clean time: 0.926s, rows: 110,013


In [ ]:
print("Rows before dropping NaNs:", len(base_df))
print(base_df[["delivery_delay_days", "product_category_name", "is_bad_review"]].isna().sum())

base_df = base_df.dropna(subset=["delivery_delay_days", "product_category_name", "is_bad_review"])
print("Rows after dropping NaNs:", len(base_df))

Rows before dropping NaNs: 110013
delivery_delay_days         8
product_category_name    1533
is_bad_review               0
dtype: int64
Rows after dropping NaNs: 108472


In [ ]:
# Group orders by product category and calculate:
    # - Average delivery delay
    # - Percentage of bad reviews
    # - Number of orders
def aggregate(df):
    return df.groupby("product_category_name").agg(
        avg_delivery_delay_days=("delivery_delay_days", "mean"),
        bad_review_rate=("is_bad_review", "mean"),
        n_orders=("order_id", "count"),
    ).reset_index().sort_values("bad_review_rate", ascending=False)
# Apply the aggregation function to our cleaned dataset
agg_result = aggregate(base_df)
agg_result.to_csv("pandas_aggregation_by_category.csv", index=False)
agg_result

,product_category_name,avg_delivery_delay_days,bad_review_rate,n_orders
42,fraldas_higiene,-10.658656,0.513514,37
67,seguros_e_servicos,-16.278264,0.500000,2
65,portateis_cozinha_e_preparadores_de_alimentos,-8.765804,0.500000,14
55,moveis_escritorio,-11.132146,0.396635,1664
15,casa_conforto_2,-7.309357,0.370370,27
...,...,...,...,...
47,livros_importados,-10.429551,0.122807,57
48,livros_interesse_geral,-11.471831,0.105263,532
22,construcao_ferramentas_ferramentas,-11.817458,0.090909,99
17,cds_dvds_musicais,-16.097030,0.071429,14


In [ ]:
# The stress test checks how well the aggregation performs
# when the dataset becomes larger.

# Create a dictionary to store our stress test and ML results
results = {"stress_test": [], "ml": {}}

# Store information about the original dataset
results["base_join_time_sec"] = join_time
results["base_rows"] = len(base_df)


def replicate(df, factor):
    # Repeat the dataset multiple times to simulate
    # a larger dataset.
    # Example: factor=5 means the dataset is repeated 5 times.
    return df if factor == 1 else pd.concat([df] * factor, ignore_index=True)


# Test the aggregation using different dataset sizes:
# 1x, 5x, 10x and 20x the original dataset
for factor in [1, 5, 10, 20]:
  # Create a larger version of the dataset
    rep_df = replicate(base_df, factor)
  # Start tracking memory usage
    tracemalloc.start()
  # Start the timer to measure processing time
    t0 = time.perf_counter()
  # Run the aggregation on the larger dataset
        # "_" means we don't need to save the output,
        # because we only want to measure performance.
    try:
        _ = aggregate(rep_df)
        # Calculate how long the aggregation took
        elapsed = time.perf_counter() - t0
        # Get the current and maximum memory usage
        current, peak = tracemalloc.get_traced_memory()
        status = "ok"
    # If the dataset is too large and the computer
    # runs out of memory, record the test as failed.
    except MemoryError:
        elapsed, peak, status = None, None, "crashed_memory_error"
    # Stop tracking memory
    tracemalloc.stop()
    print(f"factor={factor:>2} rows={len(rep_df):>10,} time={elapsed} peak_MB={None if peak is None else round(peak/1e6,1)} status={status}")
    # Save the stress test results for later comparison
    results["stress_test"].append({"factor": factor, "n_rows": len(rep_df), "time_sec": elapsed,
                                    "peak_memory_MB": None if peak is None else peak/1e6, "status": status})

factor= 1 rows=   108,472 time=0.25140776399894094 peak_MB=6.0 status=ok
factor= 5 rows=   542,360 time=0.32793683700037946 peak_MB=25.6 status=ok
factor=10 rows= 1,084,720 time=0.39420807100032107 peak_MB=51.2 status=ok
factor=20 rows= 2,169,440 time=0.7563160290010273 peak_MB=68.5 status=ok


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

X = base_df[["delivery_delay_days", "product_category_name"]]
y = base_df["is_bad_review"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# Product categories are text, so we convert them into
# numerical values using One-Hot Encoding.
pre = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), ["product_category_name"])],
                        remainder="passthrough")
# Create a pipeline:
# Step 1 → Convert product categories into numbers
# Step 2 → Train the Logistic Regression model
pipe = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=200))])
# Calculate how long the model took to train
t0 = time.perf_counter()
pipe.fit(X_train, y_train)
train_time = time.perf_counter() - t0
# ROC-AUC measures how well the model distinguishes
# between bad reviews (1) and non-bad reviews (0).
auc = roc_auc_score(y_test, pipe.predict_proba(X_test)[:, 1])
print(f"train_time={train_time:.3f}s roc_auc={auc:.4f}")
results["ml"] = {"train_time_sec": train_time, "roc_auc": auc}

train_time=5.095s roc_auc=0.6238


In [ ]:
with open("pandas_results.json", "w") as f:
    json.dump(results, f, indent=2)

from google.colab import files
files.download("pandas_results.json")
files.download("pandas_aggregation_by_category.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>